---
## ✏️ TODO — Introduction (WRITE THIS SECTION)

> **Reminder:** Write a 1–2 paragraph introduction before submitting. Cover:
> - What problem are you solving? (Content-based movie/TV recommendation)
> - Why is this interesting or useful?
> - What dataset are you using and where does it come from? (TMDB API)
> - Brief overview of your approach (TF-IDF + cosine similarity)
> - What will the reader find in this report?

---

# PSTAT 134 FINAL PROJECT
- Sid Revanuru
- Jayden Gould
- Jason Kim
- Clark Enge

## Movie Recomender System

### Pulling the data

In [ ]:
import os
import time
import pandas as pd
import requests

cache_file = "all_movie_data"

if os.path.exists(cache_file):
    movies_df = pd.read_pickle(cache_file)

else:
    API_TOKEN = # TOKEN GOES HERE
    headers = {
        "accept": "application/json",
        "Authorization": f'Bearer {API_TOKEN}'
    }
    movie_data =[]
    for page in range(1, 501):
        url = f"https://api.themoviedb.org/3/discover/movie?sort_by=popularity.desc&page={page}"
        response = requests.get(url, headers=headers)
        data = response.json()

        if "results" in data:
            for movie in data['results']:
                movie_id=movie['id']

                details_url = f"https://api.themoviedb.org/3/movie/{movie_id}?append_to_response=credits,keywords"
                details_response = requests.get(details_url, headers=headers)
                details_data = details_response.json()

                movie_dict = {
                    "id": movie_id,
                    'title': details_data['title'],
                    'overview': details_data['overview'],
                    'genres': [genre['name'] for genre in details_data.get('genres', [])],
                    'keywords': [kw['name'] for kw in details_data.get('keywords', {}).get('keywords', [])],
                    'cast': [actor['name'] for actor in details_data.get('credits', {}).get('cast', [])[:5]],
                    'vote_average': details_data['vote_average'],                  
                    'popularity': details_data['popularity']
                }

                movie_data.append(movie_dict)

                time.sleep(.10)

    movies_df = pd.DataFrame(movie_data)

    movies_df.to_pickle(cache_file)
movies_df.head()

,id,title,overview,genres,keywords,cast,vote_average,popularity
0,1226863,The Super Mario Galaxy Movie,Having thwarted Bowser's previous plot to marr...,"[Family, Comedy, Adventure, Fantasy, Animation]","[galaxy, friendship, sibling relationship, spa...","[Chris Pratt, Anya Taylor-Joy, Charlie Day, Ja...",6.732,443.9285
1,1007757,Swapped,"A small woodland creature and a majestic bird,...","[Adventure, Animation, Family, Fantasy]","[wolf, buddy, forest fire, woodlands, loving, ...","[Michael B. Jordan, Juno Temple, Tracy Morgan,...",7.892,409.3611
2,1318447,Apex,A grieving woman pushing her limits on a solo ...,"[Action, Thriller]","[canoe trip, rock climbing, nutcase, survival ...","[Charlize Theron, Taron Egerton, Eric Bana, Ca...",6.417,369.7562
3,10867,Malena,12-year-old Renato experiences three significa...,[Drama],"[prostitute, jealousy, sicily, italy, widow, w...","[Monica Bellucci, Giuseppe Sulfaro, Luciano Fe...",7.429,310.2717
4,1198994,Send Help,Two colleagues become stranded on a deserted i...,"[Horror, Thriller, Comedy]","[bullying, role reversal, survival, struggle f...","[Rachel McAdams, Dylan O'Brien, Edyll Ismail, ...",6.978,247.9714


We want this algorithm to look at large bodies of text instead different strings from different columns, causing the model to jump around. We will now take all of the text-based columns and create a new column combining all of them into one big block of text for each movie to make it easier to read for the computer.

In [2]:
import ast


# Helper function
def clean(list):
    for i in list:
       return [i.replace(" ", "") for i in list]

# Removing spaces form the values in each list column
for col in ['genres', 'cast', 'keywords']:
    movies_df[col] = movies_df[col].apply(clean)

# Turning the overview column into a list of words
movies_df['overview'] = movies_df['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Adding a new column that combines all the lists into one big list with all the information the computer needs
movies_df['tags'] = movies_df['overview'] + movies_df['genres'] + movies_df['cast'] + movies_df['keywords']

# New cleaned df with just the essential information
cleaned_movies_df = movies_df[['id', 'title', 'tags', 'vote_average', 'popularity']].copy()

# Making the giant list a string
cleaned_movies_df['tags'] = cleaned_movies_df['tags'].apply(lambda x: " ".join([str(i) for i in x]) if isinstance(x, list) else "")
# Making all the values in the giant string lowercase
cleaned_movies_df['tags'] = cleaned_movies_df['tags'].apply(lambda x: x.lower())

cleaned_movies_df.head()

,id,title,tags,vote_average,popularity
0,1226863,The Super Mario Galaxy Movie,having thwarted bowser's previous plot to marr...,6.732,443.9285
1,1007757,Swapped,"a small woodland creature and a majestic bird,...",7.892,409.3611
2,1318447,Apex,a grieving woman pushing her limits on a solo ...,6.417,369.7562
3,10867,Malena,12-year-old renato experiences three significa...,7.429,310.2717
4,1198994,Send Help,two colleagues become stranded on a deserted i...,6.978,247.9714


Next, we want to  make it so that the computer is actually able to detect the similarities/differences. To acheive this, we are going to use Term Frequency-Inverse Document Frequency. This counts how many times a word shows up in the tag, but it penalizes words that show up too much.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Vectorize the tags column and create a giant matrix
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

tfidf_matrix =tfidf.fit_transform(cleaned_movies_df['tags'])


(9960, 5000)


The `tfidf_matrix` variable returns a giant matrix of numbers that the computer can read.

Our next step is to scale the continuous variables so that large numbers will not completely break the similarity score.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler =MinMaxScaler()

# Scale the continuous variables
cleaned_movies_df[['popularity', 'vote_average']] = scaler.fit_transform(cleaned_movies_df[['popularity', 'vote_average']])

cleaned_movies_df.head()


,id,title,tags,vote_average,popularity
0,1226863,The Super Mario Galaxy Movie,having thwarted bowser's previous plot to marr...,0.6732,1.000000
1,1007757,Swapped,"a small woodland creature and a majestic bird,...",0.7892,0.921934
2,1318447,Apex,a grieving woman pushing her limits on a solo ...,0.6417,0.832491
3,10867,Malena,12-year-old renato experiences three significa...,0.7429,0.698153
4,1198994,Send Help,two colleagues become stranded on a deserted i...,0.6978,0.557455


Now its time to compute the similarity score. We will now calculate the cosine similarity matrix for the text data.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# We want to compare each movie to every other movie
cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cos_sim.shape)

(9960, 9960)


---
## 📊 TODO — Exploratory Data Analysis & Visualizations (ADD ≥ 4 PLOTS)

> **Reminder:** The rubric requires at least **4 different plots or tables**. Add visualization code cells below this reminder. Suggested plots:
>
> 1. **Genre distribution** — Bar chart of top N genres across the dataset
>    ```python
>    import matplotlib.pyplot as plt
>    movies_df['genres'].explode().value_counts().head(15).plot(kind='bar')
>    plt.title('Top 15 Genres'); plt.tight_layout(); plt.show()
>    ```
> 2. **Popularity distribution** — Histogram of popularity scores
> 3. **Vote average distribution** — Histogram or box plot of vote averages
> 4. **Word cloud or top keywords** — Most frequent words in the `tags` column
> 5. *(Optional)* Sample recommendation output table for 2–3 test inputs

---

Now its time to create the recomender function. We want to make the continous variables play a part so we are going to add weights to the popularity and vote_average so that the text similarity isnt the only factor in the recomendation. This way, if two movies are equally similar, the one with the higher popularity would have a higher similarity score.

In [ ]:
def recomend(title, text_weight=0.7, pop_weight=0.2, vote_weight=0.1):
    # FUNCTION GOES HERE
    return

---
## 📋 TODO — Results (WRITE THIS SECTION)

> **Reminder:** After completing the `recomend()` function above, write a Results section here. Include:
> - Show sample output: call `recomend()` on 2–3 test titles and display the results
> - Do recommendations stay within the same genre? Are they intuitive?
> - How do the `text_weight`, `pop_weight`, and `vote_weight` parameters affect results?
> - Compare a few outputs to what TMDB itself recommends for the same titles

---

---
## 💬 TODO — Discussion & Conclusion (WRITE THIS SECTION)

> **Reminder:** Write a 1–2 paragraph conclusion. Cover:
> - Summary of what was built and what it does
> - Key findings: what patterns did you observe in the recommendations?
> - Limitations: What does content-based filtering miss? (e.g., no user feedback, semantic gaps in overviews)
> - Future work: How could the system be improved? (e.g., collaborative filtering, embeddings, hybrid approach)

---

## ✅ Pre-Submission Checklist

Before submitting, verify all of the following:

- [ ] `recomend()` function is fully implemented and produces correct output
- [ ] Introduction section is written above
- [ ] At least **4 visualizations** are present (EDA section)
- [ ] Results section shows sample outputs and discusses findings
- [ ] Discussion & Conclusion section is complete
- [ ] **Kernel → Restart & Run All** — all cells show sequential execution numbers
- [ ] **File → Download as → HTML (.html)** → save as `PSTAT 134 FINAL PROJECT.html` and commit to repo
- [ ] README.md updated with final results summary
